# Dataproc vs Dataflow — Side-by-Side Demo

Companion notebook to the **Dataproc vs Dataflow Decision Framework Handbook**.

This notebook builds the *same* simple ETL job twice — once as a PySpark job submitted to Dataproc Serverless, once as an Apache Beam pipeline run on Dataflow — timing each and comparing the results side by side. The point isn't which one is 'better'; it's seeing the two operational experiences directly.

Run the cells top to bottom. Update the variables in the **Configuration** cell first.


## 0. Setup: install & authenticate


In [ ]:
!pip install --quiet google-cloud-dataproc google-cloud-bigquery google-cloud-storage 'apache-beam[gcp]'


In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab.')
except ImportError:
    print('Not running in Colab -- make sure you have run `gcloud auth application-default login` locally.')


## 1. Configuration — edit before running


In [ ]:
PROJECT_ID = "himanshugcpproject1"
REGION = "us-east1"
BUCKET = f"{PROJECT_ID}-decision-demo"
SOURCE_CSV = f"gs://{BUCKET}/orders/employees.csv"
SPARK_TABLE = f"{PROJECT_ID}.hr_demo.engineering_via_spark"
DATAFLOW_TABLE = f"{PROJECT_ID}.hr_demo.engineering_via_dataflow"

print(f"Project: {PROJECT_ID} | Region: {REGION} | Bucket: {BUCKET}")


## 1a. Enable required APIs


In [ ]:
import subprocess

apis = [
    "dataproc.googleapis.com",
    "dataflow.googleapis.com",
    "bigquery.googleapis.com",
    "storage.googleapis.com",
]

result = subprocess.run(
    ["gcloud", "services", "enable", *apis, f"--project={PROJECT_ID}"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("APIs enabled (or already were). Allow a minute for this to fully propagate.")


## 1b. Fix Dataflow IAM permissions (one-time)

Dataflow needs the Dataflow Service Agent to be able to act as the worker service account. This is normally auto-granted when the Dataflow API is enabled, but the grant can lag a few minutes on a freshly-enabled project, or occasionally needs the `serviceAccountUser` binding added explicitly. This cell grants everything needed in one pass; every binding here is idempotent, so it's safe to re-run.


In [ ]:
import subprocess

USER_EMAIL = "himanshu.rathi@talktotechie.com"  # <-- the account submitting the Dataflow job

def run(cmd, label):
    result = subprocess.run(cmd, capture_output=True, text=True)
    status = 'OK' if result.returncode == 0 else 'FAILED'
    print(f"[{status}] {label}")
    if result.returncode != 0:
        print(result.stderr[-1000:])
    return result

# Get the project number (needed to build service account emails)
proj = subprocess.run(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
    capture_output=True, text=True,
)
PROJECT_NUMBER = proj.stdout.strip()
print(f"Project number: {PROJECT_NUMBER}")

run(
    ["gcloud", "services", "enable", "dataflow.googleapis.com", f"--project={PROJECT_ID}"],
    "Enable Dataflow API",
)

run(
    [
        "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
        f"--member=serviceAccount:service-{PROJECT_NUMBER}@dataflow-service-producer-prod.iam.gserviceaccount.com",
        "--role=roles/dataflow.serviceAgent", "--condition=None",
    ],
    "Grant Dataflow Service Agent role",
)

run(
    [
        "gcloud", "iam", "service-accounts", "add-iam-policy-binding",
        f"{PROJECT_NUMBER}[email protected]",
        f"--member=user:{USER_EMAIL}",
        "--role=roles/iam.serviceAccountUser", "--condition=None",
    ],
    "Grant serviceAccountUser on the worker service account",
)

run(
    [
        "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
        f"--member=user:{USER_EMAIL}",
        "--role=roles/dataflow.admin", "--condition=None",
    ],
    "Grant dataflow.admin",
)

run(
    [
        "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
        f"--member=user:{USER_EMAIL}",
        "--role=roles/dataflow.worker", "--condition=None",
    ],
    "Grant dataflow.worker",
)

print("\nDone. Wait 2-3 minutes for IAM propagation before running the Dataflow cell below.")


## 2. Create the bucket and export sample CSV data


In [ ]:
from google.cloud import storage, bigquery

storage_client = storage.Client(project=PROJECT_ID)
bq_client = bigquery.Client(project=PROJECT_ID)

try:
    bucket = storage_client.create_bucket(BUCKET, location=REGION)
    print(f"Bucket created: {bucket.name}")
except Exception as e:
    print(f"Bucket may already exist: {e}")


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)

# Create the dataset
dataset_ref = bigquery.DatasetReference(PROJECT_ID, "hr_demo")
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "us"
bq_client.create_dataset(dataset, exists_ok=True)

# Create the table
schema = [
    bigquery.SchemaField("employee_id", "INT64"),
    bigquery.SchemaField("full_name", "STRING"),
    bigquery.SchemaField("department", "STRING"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("salary", "NUMERIC"),
]
table_ref = bigquery.TableReference(dataset_ref, "employees")
table = bigquery.Table(table_ref, schema=schema)
bq_client.create_table(table, exists_ok=True)

# Insert sample rows
rows = [
    {"employee_id": 1, "full_name": "Riya Sharma", "department": "Engineering", "email": "[email protected]", "salary": 950000},
    {"employee_id": 2, "full_name": "Arjun Verma", "department": "Sales", "email": "[email protected]", "salary": 720000},
    {"employee_id": 3, "full_name": "Meera Nair", "department": "Finance", "email": "[email protected]", "salary": 880000},
]
errors = bq_client.insert_rows_json(table, rows)
print("Insert errors:", errors if errors else "none")

In [ ]:
# Export hr_demo.employees to CSV for both pipelines to read
extract_job = bq_client.extract_table(
    f"{PROJECT_ID}.hr_demo.employees",
    SOURCE_CSV,
)
extract_job.result()
print(f"Exported employees table to {SOURCE_CSV}")


## 3. Run A: Dataproc Serverless (PySpark)


In [ ]:
spark_script = f"""
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DecisionDemoSpark").getOrCreate()

df = spark.read.option("header", True).csv("{SOURCE_CSV}")
filtered = df.filter(col("department") == "Engineering")

filtered.write.format("bigquery") \\
    .option("table", "{SPARK_TABLE}") \\
    .option("writeMethod", "direct") \\
    .mode("overwrite") \\
    .save()
"""

with open("spark_etl.py", "w") as f:
    f.write(spark_script)

bucket_obj = storage_client.bucket(BUCKET)
blob = bucket_obj.blob("scripts/spark_etl.py")
blob.upload_from_filename("spark_etl.py")
SPARK_SCRIPT_URI = f"gs://{BUCKET}/scripts/spark_etl.py"
print(f"Uploaded {SPARK_SCRIPT_URI}")


In [ ]:
import time
from google.cloud import dataproc_v1

batch_client = dataproc_v1.BatchControllerClient(
    client_options={"api_endpoint": f"{REGION}-dataproc.googleapis.com:443"}
)

batch = dataproc_v1.Batch()
batch.pyspark_batch.main_python_file_uri = SPARK_SCRIPT_URI
batch.runtime_config.version = "2.2"

parent = f"projects/{PROJECT_ID}/locations/{REGION}"

spark_start = time.time()
operation = batch_client.create_batch(parent=parent, batch=batch)
print("Spark batch submitted, waiting for completion...")
result = operation.result()
spark_elapsed = time.time() - spark_start
print(f"Spark batch finished with state {result.state} in {spark_elapsed:.1f}s")


## 4. Run B: Dataflow (Apache Beam)

This runs as a local subprocess launching the Beam pipeline with DataflowRunner — the pipeline itself executes on Dataflow workers, but the Python process here blocks until the job completes (this can take several minutes for a Dataflow job's startup + teardown).


In [ ]:
beam_script = f"""
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

options = PipelineOptions([
    "--runner=DataflowRunner",
    "--project={PROJECT_ID}",
    "--region={REGION}",
    "--temp_location=gs://{BUCKET}/temp/",
])

with beam.Pipeline(options=options) as p:
    (
        p
        | "ReadCSV" >> beam.io.ReadFromText("{SOURCE_CSV}", skip_header_lines=1)
        | "ParseCSV" >> beam.Map(lambda line: dict(zip(
              ["employee_id", "full_name", "department", "email", "salary"], line.split(","))))
        | "FilterEngineering" >> beam.Filter(lambda r: r["department"] == "Engineering")
        | "WriteBQ" >> beam.io.WriteToBigQuery(
              "{DATAFLOW_TABLE}",
              schema="employee_id:STRING,full_name:STRING,department:STRING,email:STRING,salary:STRING",
              write_disposition=beam.io.BigQueryDisposition.WRITE_TRUNCATE)
    )
"""

with open("beam_etl.py", "w") as f:
    f.write(beam_script)

print("Wrote beam_etl.py")


In [ ]:
import subprocess

dataflow_start = time.time()
proc = subprocess.run(["python", "beam_etl.py"], capture_output=True, text=True)
dataflow_elapsed = time.time() - dataflow_start

print(proc.stdout[-2000:])
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-2000:])
print(f"Dataflow job process finished in {dataflow_elapsed:.1f}s (includes local launch overhead)")


## 5. Compare the results


In [ ]:
for label, table, elapsed in [
    ("Dataproc Serverless (Spark)", SPARK_TABLE, spark_elapsed),
    ("Dataflow (Beam)", DATAFLOW_TABLE, dataflow_elapsed),
]:
    try:
        t = bq_client.get_table(table)
        print(f"{label:32s} | rows={t.num_rows:4d} | wall_time={elapsed:7.1f}s")
    except Exception as e:
        print(f"{label:32s} | Error: {e}")

print("\nRow counts should match exactly -- both pipelines apply the same filter to the same source data.")
print("The interesting comparison is qualitative: check the Dataproc Batches console vs the Dataflow")
print("Jobs console (with its live job graph) to see the two operational experiences directly.")


## 6. Cleanup


In [ ]:
CONFIRM_CLEANUP = True  # flip to True to actually delete resources

if CONFIRM_CLEANUP:
    bq_client.delete_table(SPARK_TABLE, not_found_ok=True)
    bq_client.delete_table(DATAFLOW_TABLE, not_found_ok=True)
    blobs = list(bucket_obj.list_blobs())
    for b in blobs:
        b.delete()
    bucket_obj.delete()
    print("Output tables and demo bucket deleted.")
    print("Remember to also check for any lingering Dataflow jobs:")
    print(f"  gcloud dataflow jobs list --region={REGION}")
else:
    print("Skipped. Set CONFIRM_CLEANUP = True and re-run this cell to clean up.")
